# Supervised Learning & Optimization: Logistic Regression



In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import StratifiedKFold, GridSearchCV

# 1. Load Medical Dataset
data = load_breast_cancer()
X = data.data
y = data.target
print(f"Dataset loaded: {X.shape}")

# 2. Z-Score Standardization (Programmed from scratch)
def custom_z_score(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0) + 1e-9  # Add epsilon to prevent division by zero
    return (X - mu) / sigma

X_scaled = custom_z_score(X)
print(f"Manual Z-Score Standardization applied: Mean = {np.mean(X_scaled):.4f}, Std = {np.mean(np.std(X_scaled, axis=0)):.4f}")

# 3. Set up Stratified K-Fold to maintain class distribution
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
precisions, recalls, f1_scores = [], [], []

print("-" * 50)
print("K-FOLD CROSS VALIDATION RESULTS (5 Folds)")
print("-" * 50)

# 4. Manual K-Fold CV Loop with Systematic Metric Evaluation
fold = 1
for train_index, val_index in kf.split(X_scaled, y):
    X_train, X_val = X_scaled[train_index], X_scaled[val_index]
    y_train, y_val = y[train_index], y[val_index]
    
    # Train logistic regression model
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    
    # Predict and systematically evaluate
    y_pred = model.predict(X_val)
    p = precision_score(y_val, y_pred)
    r = recall_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    
    precisions.append(p)
    recalls.append(r)
    f1_scores.append(f1)
    
    print(f"Fold {fold} - Precision: {p:.4f}, Recall: {r:.4f}, F1: {f1:.4f}")
    fold += 1

print("\nAVERAGE METRICS ACROSS ALL FOLDS:")
print(f"Average Precision: {np.mean(precisions):.4f}")
print(f"Average Recall:    {np.mean(recalls):.4f}")
print(f"Average F1-Score:  {np.mean(f1_scores):.4f}")

Dataset loaded: (569, 30)
Manual Z-Score Standardization applied: Mean = -0.0000, Std = 1.0000
--------------------------------------------------
K-FOLD CROSS VALIDATION RESULTS (5 Folds)
--------------------------------------------------
Fold 1 - Precision: 0.9857, Recall: 0.9718, F1: 0.9787
Fold 2 - Precision: 0.9221, Recall: 1.0000, F1: 0.9595
Fold 3 - Precision: 0.9474, Recall: 1.0000, F1: 0.9730
Fold 4 - Precision: 1.0000, Recall: 0.9861, F1: 0.9930
Fold 5 - Precision: 0.9861, Recall: 1.0000, F1: 0.9930

AVERAGE METRICS ACROSS ALL FOLDS:
Average Precision: 0.9683
Average Recall:    0.9916
Average F1-Score:  0.9794


### Grid Search Optimization
Aggressively mitigating model overfitting by finding the optimal L2 regularization strength (C parameter) across the full feature space.

In [2]:
# 5. Grid Search for Hyperparameter Tuning
param_grid = {
    'C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    'penalty': ['l2']
}

# Execute Grid Search using 5-Fold CV optimized for F1-score
grid_search = GridSearchCV(
    LogisticRegression(max_iter=1000), 
    param_grid, 
    cv=5, 
    scoring='f1', 
    n_jobs=-1
)
grid_search.fit(X_scaled, y)

print(f"Optimal Hyperparameters to mitigate overfitting: {grid_search.best_params_}")
print(f"Optimized Cross-Validated F1-Score: {grid_search.best_score_:.4f}")

# 6. Extract the final optimized model
final_model = grid_search.best_estimator_
print("Final tuned model is successfully trained on the full dataset and ready for deployment.")

Optimal Hyperparameters to mitigate overfitting: {'C': 1.0, 'penalty': 'l2'}
Optimized Cross-Validated F1-Score: 0.9848
Final tuned model is successfully trained on the full dataset and ready for deployment.
